# Certify-or-Grow with real Q-learning, not exact linear algebra

Every other notebook in this repo solves the metapolicy exactly (dense linear-algebra
policy iteration). In practice, nobody does that -- SFOLS/GPI implementations
(`RM/sf-fsa-vi/sfols/rl/successor_features/tabular_sf.py`) learn $\Psi$ online via TD.
This notebook replaces exact policy iteration with the *same* SFQL-style TD update used
in that vendored codebase, restricted to $B(s)$ (Definition 2's restricted Bellman
operator) instead of the full action set, and tests it on DST (`mo_gymnasium`).

**Question:** does restricting a Q-learner's behavior AND bootstrap to $B(s)$ (the
already-trained base policies' own actions) converge faster than training an
unrestricted Q-learner from scratch, for a new query weight?

**Result:** yes, decisively. The restricted metapolicy reaches the exact optimum with as
few as 250 environment steps; an unrestricted from-scratch learner given the identical
schedule needs 1000 -- a $4\times$ reduction in per-query environment interaction,
*after* the (amortized, one-time) cost of building the $k$-policy base pool.

## 1. A tabular SFQL agent (matches `tabular_sf.py`'s update rule exactly)

In [1]:
import numpy as np
import mo_gymnasium as mo_gym

GAMMA = 0.98
ALPHA = 0.1
FEAT_DIM = 2
N_ACTIONS = 4


def obs_key(obs):
    return (int(obs[0]), int(obs[1]))


def make_env():
    return mo_gym.make("deep-sea-treasure-v0")


class SFAgent:
    """Tabular successor-feature Q-learner, matching the update rule in
    RM/sf-fsa-vi/sfols/rl/successor_features/tabular_sf.py's SF.train: vector-valued TD
    learning of Psi(s,a), bootstrapped via the scalarized argmax at the query weight."""

    def __init__(self, action_dim=N_ACTIONS, feat_dim=FEAT_DIM, alpha=ALPHA, gamma=GAMMA):
        self.q_table = {}
        self.action_dim = action_dim
        self.feat_dim = feat_dim
        self.alpha = alpha
        self.gamma = gamma

    def _row(self, s):
        if s not in self.q_table:
            self.q_table[s] = np.zeros((self.action_dim, self.feat_dim))
        return self.q_table[s]

    def q_values(self, s, w):
        return self._row(s) @ w

    def act_eps_greedy(self, s, w, eps, rng, allowed=None):
        allowed = allowed if allowed is not None else list(range(self.action_dim))
        if rng.random() < eps:
            return int(rng.choice(allowed))
        qv = self.q_values(s, w)
        masked = np.full(self.action_dim, -np.inf)
        masked[allowed] = qv[allowed]
        return int(np.argmax(masked))

    def update(self, s, a, phi, s_next, terminal, w, allowed_next=None):
        row = self._row(s)
        if terminal:
            target = phi
        else:
            allowed_next = allowed_next if allowed_next is not None else list(range(self.action_dim))
            next_row = self._row(s_next)
            qv_next = next_row @ w
            masked = np.full(self.action_dim, -np.inf)
            masked[allowed_next] = qv_next[allowed_next]
            a_star = int(np.argmax(masked))
            target = phi + self.gamma * next_row[a_star]
        row[a] += self.alpha * (target - row[a])


def linear_epsilon(t, warmup, decay_steps, eps0=1.0, eps1=0.05):
    if t < warmup:
        return eps0
    frac = min(1.0, (t - warmup) / max(1, decay_steps))
    return eps0 + frac * (eps1 - eps0)


def train_sf_agent(env, w, total_steps, warmup, decay_steps, seed, allowed_fn=None):
    """allowed_fn(s) -> list of allowed actions at s. None = full action set (standard
    SFQL). A function returning B(s) = the k base policies' own greedy actions at s
    turns this into the restricted metapolicy learner of Definition 2, learned online."""
    rng = np.random.default_rng(seed)
    agent = SFAgent()
    obs, _ = env.reset(seed=seed)
    s = obs_key(obs)
    for t in range(total_steps):
        eps = linear_epsilon(t, warmup, decay_steps)
        allowed = allowed_fn(s) if allowed_fn else None
        a = agent.act_eps_greedy(s, w, eps, rng, allowed=allowed)
        obs_next, r_vec, terminated, truncated, _ = env.step(a)
        s_next = obs_key(obs_next)
        allowed_next = allowed_fn(s_next) if allowed_fn else None
        agent.update(s, a, np.array(r_vec, dtype=float), s_next, terminated, w, allowed_next=allowed_next)
        if terminated or truncated:
            obs, _ = env.reset()
            s = obs_key(obs)
        else:
            s = s_next
    return agent


def greedy_rollout_value(env, agent, w, gamma, max_steps=200, allowed_fn=None):
    obs, _ = env.reset(seed=0)
    s = obs_key(obs)
    total, disc = 0.0, 1.0
    for _ in range(max_steps):
        allowed = allowed_fn(s) if allowed_fn else list(range(N_ACTIONS))
        qv = agent.q_values(s, w)
        masked = np.full(N_ACTIONS, -np.inf)
        masked[allowed] = qv[allowed]
        a = int(np.argmax(masked))
        obs, r_vec, terminated, truncated, _ = env.step(a)
        total += disc * float(np.array(r_vec) @ w)
        disc *= gamma
        s = obs_key(obs)
        if terminated or truncated:
            break
    return total


def own_greedy_action(agent, s, w_train):
    row = agent._row(s)
    return int(np.argmax(row @ w_train))


## 2. Ground truth: exact optimal value at the query weight (BFS + value iteration)

In [2]:
def exact_optimal_value(w, gamma, max_states=2000):
    """BFS-discover DST's deterministic transition graph, then exact value iteration --
    the ground truth this cell's Q-learning results are checked against."""
    env = make_env()
    obs, _ = env.reset(seed=0)
    start = tuple(int(x) for x in obs)
    trans = {}
    frontier = [start]
    seen = {start}
    while frontier and len(seen) < max_states:
        s = frontier.pop()
        for a in range(N_ACTIONS):
            e2 = make_env()
            e2.reset(seed=0)
            e2.unwrapped.current_state = np.array(s)
            obs2, r, term, trunc, _ = e2.step(a)
            s2 = tuple(int(x) for x in obs2)
            trans[(s, a)] = (s2, np.array(r, dtype=float), term)
            if s2 not in seen:
                seen.add(s2)
                frontier.append(s2)
    states = list(seen)
    idx = {s: i for i, s in enumerate(states)}
    n = len(states)
    V = np.zeros(n)
    for _ in range(200):
        newV = np.zeros(n)
        for s in states:
            qs = [trans[(s, a)][1] @ w + (0.0 if trans[(s, a)][2] else gamma * V[idx[trans[(s, a)][0]]])
                  for a in range(N_ACTIONS) if (s, a) in trans]
            newV[idx[s]] = max(qs) if qs else 0.0
        if np.max(np.abs(newV - V)) < 1e-10:
            V = newV
            break
        V = newV
    return V[idx[start]]


w_query = np.array([0.4, 0.6])
v_star = exact_optimal_value(w_query, GAMMA)
print(f"Exact optimal scalarized value at the start state, w={w_query}: {v_star:.4f}")


/opt/anaconda3/envs/okb/lib/python3.10/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


Exact optimal scalarized value at the start state, w=[0.4 0.6]: 1.3859


## 3. Train the $k=2$ base pool once (canonical corner weights, online TD)

This is the one-time cost, exactly analogous to the exact-solve base policies used
everywhere else in this repo -- amortized over every future query weight.

In [3]:
import time

env = make_env()
w0, w1 = np.array([1.0, 0.0]), np.array([0.0, 1.0])
BASE_STEPS, BASE_WARMUP, BASE_DECAY = 8000, 1000, 4000

t0 = time.perf_counter()
base0 = train_sf_agent(env, w0, BASE_STEPS, BASE_WARMUP, BASE_DECAY, seed=0)
base1 = train_sf_agent(env, w1, BASE_STEPS, BASE_WARMUP, BASE_DECAY, seed=1)
t_base = time.perf_counter() - t0
print(f"k=2 base SFQL agents trained via online TD (canonical corners e0, e1): "
      f"{2*BASE_STEPS} total env steps, {t_base:.1f}s wall clock.")
print("This cost is paid ONCE and amortized across every future query weight --")
print("exactly the same accounting as the exact-solve Certify-or-Grow speedups above.")


def B_of(s):
    a0 = own_greedy_action(base0, s, w0)
    a1 = own_greedy_action(base1, s, w1)
    return sorted(set([a0, a1]))


k=2 base SFQL agents trained via online TD (canonical corners e0, e1): 16000 total env steps, 4.5s wall clock.
This cost is paid ONCE and amortized across every future query weight --
exactly the same accounting as the exact-solve Certify-or-Grow speedups above.


## 4. Per-query budget sweep: restricted metapolicy vs. unrestricted from-scratch

Same epsilon schedule, same learning rate, same seeds -- the only difference is whether
the action set at each step is restricted to $B(s)$ or left as the full action set.

In [4]:
budgets = [50, 100, 150, 200, 250, 400, 600, 1000]
seeds = list(range(10, 20))

print(f"Per-query training budget sweep, 10 seeds each, query weight w={w_query}:\n")
print(f"{'budget':>7} {'meta mean':>10} {'meta gap':>9} {'scratch mean':>13} {'scratch gap':>12}")
rows = []
for budget in budgets:
    meta_vals, scratch_vals = [], []
    for seed in seeds:
        warmup, decay = budget // 8, budget // 2
        m_agent = train_sf_agent(env, w_query, budget, warmup, decay, seed=seed, allowed_fn=B_of)
        s_agent = train_sf_agent(env, w_query, budget, warmup, decay, seed=seed, allowed_fn=None)
        meta_vals.append(greedy_rollout_value(env, m_agent, w_query, GAMMA, allowed_fn=B_of))
        scratch_vals.append(greedy_rollout_value(env, s_agent, w_query, GAMMA, allowed_fn=None))
    mm, sm = float(np.mean(meta_vals)), float(np.mean(scratch_vals))
    rows.append((budget, mm, v_star - mm, sm, v_star - sm))
    print(f"{budget:>7} {mm:>10.4f} {v_star-mm:>9.4f} {sm:>13.4f} {v_star-sm:>12.4f}")

print("\nsteps for the mean gap to first drop below 0.05 (i.e. essentially converged):")
for label, col in [("restricted metapolicy", 2), ("unrestricted from-scratch", 4)]:
    hit = next((r[0] for r in rows if r[col] < 0.05), None)
    print(f"  {label:26s}: {hit}")


Per-query training budget sweep, 10 seeds each, query weight w=[0.4 0.6]:

 budget  meta mean  meta gap  scratch mean  scratch gap


     50    -0.3200    1.7059       -2.8901       4.2760


    100    -0.3200    1.7059       -0.3200       1.7059


    150     0.1918    1.1941       -0.3200       1.7059


    200     0.7035    0.6823       -0.3200       1.7059


    250     1.3859   -0.0000       -0.3200       1.7059


    400     1.3859   -0.0000       -0.3200       1.7059


    600     1.3859   -0.0000        0.1918       1.1941


   1000     1.3859   -0.0000        1.3859      -0.0000

steps for the mean gap to first drop below 0.05 (i.e. essentially converged):
  restricted metapolicy     : 250
  unrestricted from-scratch : 1000


## Conclusion

Restricting a TD learner's behavior and bootstrap to $B(s)$ is a real, practical
implementation of Certify-or-Grow's metapolicy -- not just an exact-linear-algebra
abstraction. On DST it converges in a quarter of the environment steps an unrestricted
learner needs for the same query weight, consistent in kind with the exact-solve
speedups reported earlier ($1.78\times$ DST, $13.64\times$ Office) and the GPI-init
warm-start result ($8.33\times$). As with those, the win is genuinely a *per-query* one:
building the $k$-policy base pool here cost $16{,}000$ environment steps up front, so this
only pays off once a handful of query weights are asked -- the same amortization argument
already made for the exact-solve algorithms.